# DOFA + lc-col BigEarthNet Real-Data Smoke Run

Colab-first template for the first real Sentinel-2 BigEarthNet smoke result using `lc-col/bigearthnet` and DOFA torch.hub mode. No fake chips are generated.


In [ ]:
# 1. Clone or update this repo
# Edit this if your fork/repo URL differs.
REPO_URL = "https://github.com/strivekboy-coder/rsfm-fairness-audit.git"
REPO_DIR = "rsfm-fairness-audit"

from pathlib import Path
if Path(REPO_DIR).exists():
    %cd {REPO_DIR}
    !git pull
else:
    !git clone {REPO_URL}
    %cd {REPO_DIR}


In [ ]:
# 2. Install the package
!python -m pip install -e .


In [ ]:
# 3. Install DOFA + Hugging Face/HDF5 dependencies
!python -m pip install -r requirements-dofa.txt


In [ ]:
# 4. Check GPU
import torch
print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


In [ ]:
# Configure DOFA for current torch.hub mode.
# This explicitly allows downloading the official DOFA checkpoint when run-real first loads the model.
from pathlib import Path
import yaml

PREPARED_DATA_ROOT = "data/bigearthnet_lccol_subset"
OUTPUT_DIR = "outputs/dofa_bigearthnet_lccol64"
config_path = Path("configs/models/dofa.yaml")
config = yaml.safe_load(config_path.read_text())
config["repo_path"] = None
config["checkpoint_path"] = None
config["allow_torch_hub_download"] = True
config["device"] = "auto"
config_path.write_text(yaml.safe_dump(config, sort_keys=False))
print(config_path.read_text())


## Real lc-col BigEarthNet Data

The next cell downloads one real `lc-col/bigearthnet` HDF5 train shard from Hugging Face, inspects its HDF5 keys, and converts the first 64 real Sentinel-2 chips to this project's adapter format. The shard is large, so run this in Colab rather than committing data to Git.


In [ ]:
# 5. Download one real lc-col/bigearthnet HDF5 shard and convert 64 real Sentinel-2 chips
!python scripts/download_bigearthnet_lccol_subset.py \
  --output-dir {PREPARED_DATA_ROOT} \
  --max-samples 64 \
  --seed 42


In [ ]:
# 6. Run preflight checker
!python -m rsfm_fairness_audit.cli check-real \
  --model dofa \
  --dataset bigearthnet \
  --model-config configs/models/dofa.yaml \
  --data-root {PREPARED_DATA_ROOT}


In [ ]:
# 7. Run real DOFA + lc-col BigEarthNet smoke test
!python -m rsfm_fairness_audit.cli run-real \
  --dataset bigearthnet \
  --dataset-root {PREPARED_DATA_ROOT} \
  --model dofa \
  --config configs/models/dofa.yaml \
  --output-dir {OUTPUT_DIR} \
  --max-samples 64


In [ ]:
# 8. Inspect outputs
!find {OUTPUT_DIR} -maxdepth 3 -type f -print
!sed -n '1,160p' {OUTPUT_DIR}/report.md

from IPython.display import Image, display
for fig in [f"{OUTPUT_DIR}/figures/average_vs_worst_group.png", f"{OUTPUT_DIR}/figures/fairness_map.png"]:
    if Path(fig).exists():
        display(Image(filename=fig))


In [ ]:
# 9. Zip outputs for download
!zip -r dofa_bigearthnet_lccol64_outputs.zip {OUTPUT_DIR}
from google.colab import files
files.download("dofa_bigearthnet_lccol64_outputs.zip")
